# Семинар 2. NumPy: массивы, индексация, векторизация

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-02.ipynb`, например `2MP9-Ivanov-Ivan-seminar-02.ipynb`.
- Порядок выполнения важен: данные картотеки создаются в разделе 2 и используются до конца ноутбука.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-02-tasks.md` с формулировками и ноутбук `seminar-02-tasks.ipynb` с заготовками и тестами. Ответы вводятся в тест Moodle «Семинар 2».
- Шпаргалка по функциям: `reference/numpy.md`.

## 1. Зачем NumPy юристу

Сюжет семинара — картотека дел одного судьи. Начнём с того, что уже умеем: цены десяти исков лежат в списке Python. Нужны средняя цена и сумма каждого иска после уменьшения на 30 %: истцы снизили требования. Со списком любое действие над всеми элементами — это цикл, явный или в виде генератора списка.

In [ ]:
amounts_list = [48_000, 250_000, 1_200_000, 75_000, 320_000, 15_000, 640_000, 98_000, 410_000, 135_000]

sum(amounts_list) / len(amounts_list)   # средняя цена иска

In [ ]:
[amount * 0.7 for amount in amounts_list]   # цены после скидки 30 %

Библиотека NumPy хранит такие данные в **массиве** (`ndarray`): наборе элементов одного типа, над которым операции выполняются сразу целиком. Массив создаётся из списка функцией `np.array`; те же два расчёта записываются без цикла.

In [ ]:
import numpy as np

amounts = np.array(amounts_list)
amounts

In [ ]:
amounts.mean()   # средняя цена иска

In [ ]:
amounts * 0.7    # цены после скидки 30 %

**Векторизация** — запись операции для всего массива без цикла по элементам: `amounts * 0.7` вместо генератора списка. Код короче, читается как формула и работает быстрее: цикл выполняется внутри NumPy, на языке C. Таблицы pandas, с которыми мы начнём работать на семинаре 4, устроены из таких массивов: каждый столбец таблицы — массив.

## 2. Создание и устройство массива

У массива есть **тип данных** `dtype`, общий для всех элементов; NumPy подбирает его по содержимому списка. Целые числа дают `int64`. Одно дробное число среди целых делает дробными все элементы: тип `float64`. Строки получают тип `U` (Unicode) с числом, равным длине самой длинной строки. Значения `True` и `False` дают логический тип `bool`.

In [ ]:
claims = np.array([48_000, 250_000, 1_200_000])
claims.dtype

In [ ]:
np.array([48_000.50, 250_000, 1_200_000]).dtype

In [ ]:
np.array(['А40', 'А56', 'А41']).dtype

In [ ]:
np.array([True, False, True]).dtype

Тип меняется методом `astype`, он возвращает новый массив. При переводе дробных чисел в целые дробная часть отбрасывается, а не округляется: 4.9 превращается в 4, 5.5 — в 5. Это пригодится в упражнениях.

In [ ]:
claims.astype(float)

In [ ]:
np.array([4.9, 5.5]).astype(int)

Картотека на десять дел: цены исков, сроки рассмотрения в днях и исходы. Данные создаёт генератор случайных чисел с зерном, как в семинаре 1, поэтому у всех в аудитории получатся одни и те же числа; массив `amounts` из раздела 1 при этом заменяется картотекой. Метод `integers(от, до, size=n)` даёт `n` целых чисел; правая граница не включается: цены от 10 000 до 999 999 рублей, сроки от 10 до 199 дней, исходы 0 или 1.

In [ ]:
rng = np.random.default_rng(2026)

amounts = rng.integers(10_000, 1_000_000, size=10)   # цена иска, руб.
days = rng.integers(10, 200, size=10)                # срок рассмотрения, дней
outcomes = rng.integers(0, 2, size=10)               # 1 удовлетворён, 0 отказано

print(amounts)
print(days)
print(outcomes)

Три свойства описывают устройство массива: `shape` — **форма**, кортеж с числом элементов по каждой оси; `ndim` — число осей; `size` — общее число элементов. У одномерного массива из десяти чисел форма `(10,)`, одна ось, десять элементов.

In [ ]:
amounts.shape, amounts.ndim, amounts.size

Две функции создают массивы без списка. `np.arange` даёт последовательность чисел, правая граница не включается, как в `range`. `np.zeros` даёт массив нулей заданной длины, дробного типа: заготовку, которую потом заполняют значениями.

In [ ]:
np.arange(1, 11)   # номера дел

In [ ]:
np.zeros(10)

## 3. Арифметика и сравнения

Арифметика с массивом выполняется **поэлементно**. Массив и число: операция применяется к каждому элементу, так цены переводятся в тысячи рублей. Два массива одной длины: операция применяется к парам элементов с одинаковыми индексами, так по каждому делу считается неустойка 0,1 % в день за время рассмотрения: цена, умноженная на 0,001 и на число дней.

In [ ]:
amounts / 1_000            # цены в тысячах рублей

In [ ]:
amounts * 0.001 * days     # неустойка за время рассмотрения, руб.

Сравнение тоже поэлементное: результат — массив из `True` и `False` той же длины, **логический массив**. В разделе 5 он станет главным инструментом отбора данных.

In [ ]:
amounts > 100_000

Массивы разной длины складывать нельзя. Ошибка ниже оставлена намеренно: прочитайте сообщение целиком. NumPy сообщает, что не смог совместить формы `(10,)` и `(3,)`; слово `broadcast` означает правила совмещения форм, из них нам пока нужно одно: длины должны совпадать.

In [ ]:
amounts + np.array([1, 2, 3])

## 4. Индексация и срезы

Элементы нумеруются с нуля, отрицательный индекс считает с конца, срез `[от:до]` не включает правую границу. Всё как у списков.

In [ ]:
amounts[0], amounts[-1]   # первое и последнее дело

In [ ]:
amounts[2:5]              # дела с третьего по пятое

In [ ]:
amounts[-3:]              # три последних дела

Отличие от списка: срез массива не копирует данные, а даёт **вид** (view) на тот же участок памяти. Изменение среза меняет исходный массив. Ниже это показано на отдельном маленьком массиве, чтобы не испортить картотеку: после записи в срез первый элемент `demo` меняется. Независимую копию даёт метод `copy`: во второй ячейке `demo` остаётся прежним.

In [ ]:
demo = np.array([10, 20, 30, 40])
part = demo[:2]
part[0] = 99
demo

In [ ]:
demo = np.array([10, 20, 30, 40])
part = demo[:2].copy()
part[0] = 99
demo

## 5. Логические маски

Логический массив из раздела 3 называют **маской**: если поставить его в квадратные скобки, останутся только элементы, напротив которых стоит `True`. Ниже маска отмечает крупные иски, дороже 100 000 рублей.

In [ ]:
mask = amounts > 100_000
mask

In [ ]:
amounts[mask]   # только крупные иски

У маски две полезные статистики: `sum` считает значения `True` как единицы и даёт число подходящих элементов, `mean` даёт их долю.

In [ ]:
mask.sum(), mask.mean()   # число и доля крупных исков

Маска по одному массиву отбирает элементы другого массива той же длины: дела те же самые, поэтому индексы совпадают.

In [ ]:
days[amounts > 100_000]   # сроки крупных исков

Условия соединяются операторами `&` (и), `|` (или), `~` (не). Каждое условие берётся в скобки: без них `&` выполнится раньше сравнения. Слова `and`, `or`, `not` с массивами не работают.

In [ ]:
amounts[(amounts > 100_000) & (outcomes == 1)]   # удовлетворённые крупные иски

In [ ]:
amounts[~mask]   # иски не дороже 100 000

In [ ]:
amounts[(amounts < 50_000) | (amounts > 500_000)]   # мелкие и очень крупные иски

Две ошибки, которые сделает каждый. Первая: условия без скобок. Вторая: `and` вместо `&`. Обе дают одно и то же сообщение про неоднозначную истинность массива: Python пытается превратить массив в одно значение `True` или `False` и не знает как.

In [ ]:
amounts[amounts > 100_000 & outcomes == 1]

In [ ]:
amounts[(amounts > 100_000) and (outcomes == 1)]

Маска работает и при присваивании: значение записывается только в отмеченные элементы. Так ограничим требования потолком возмещения по ОСАГО, 400 000 рублей; исходный массив не трогаем, работаем с копией.

In [ ]:
capped = amounts.copy()
capped[capped > 400_000] = 400_000
capped

Сравнивать можно и массив строк: условие `courts == 'А40'` отмечает дела нужного суда. Суд по каждому делу ниже выбирает генератор методом `choice`: случайный элемент из списка для каждого из десяти дел. О случайных выборках подробнее на семинаре 3.

In [ ]:
courts = rng.choice(['А40', 'А56', 'А41'], size=10)   # суд по каждому делу
print(courts)
amounts[courts == 'А40']

## 6. Статистики

Сумма, среднее, минимум и максимум — методы массива. Функция `round` округляет результат, как обычное число.

In [ ]:
amounts.sum(), amounts.min(), amounts.max()

In [ ]:
round(amounts.mean())   # средняя цена, до рубля

Среднее массива из нулей и единиц — это доля единиц: доля удовлетворённых исков считается одной строкой. Статистика по отобранным делам — это статистика по маске: ниже разница средних сроков между делами с удовлетворённым иском и делами с отказом. Медиана, квантили, стандартное отклонение и поиск позиции максимума — на семинаре 3.

In [ ]:
outcomes.mean()   # доля удовлетворённых исков

In [ ]:
days[outcomes == 1].mean() - days[outcomes == 0].mean()   # разница средних сроков, дней

## 7. Двумерный массив как таблица

Картотека целиком — это таблица: строка — дело, столбец — поле. Двумерный массив собирается из одномерных: `np.array([amounts, days, outcomes])` даёт три строки по десять чисел, а `.T` меняет строки и столбцы местами, и получается десять дел по три поля. Форма такой таблицы `(10, 3)`: две оси, десять строк, три столбца.

In [ ]:
table = np.array([amounts, days, outcomes]).T
table

In [ ]:
table.shape, table.ndim

Индексы разделяются запятой: первый — строка, второй — столбец. Двоеточие означает «все». Один индекс даёт строку, то есть дело целиком; `[:, 1]` даёт второй столбец, сроки всех дел, и по нему считается средний срок.

In [ ]:
table[0]              # первое дело

In [ ]:
table[:, 1]           # столбец сроков

In [ ]:
table[:, 1].mean()    # средний срок, дней

Статистики по таблице считаются вдоль **оси** `axis`: `axis=0` — вдоль строк, по одному числу на столбец; `axis=1` — вдоль столбцов, по одному числу на строку; без `axis` — по всей таблице. Картотека для этого плохо подходит: складывать цену, дни и исход бессмысленно. Возьмём таблицу нагрузки: три суда, четыре года, число дел за год.

In [ ]:
load = np.array([[1200, 1350, 1420, 1510],
                 [ 800,  790,  910,  880],
                 [2100, 2250, 2300, 2390]])

load.sum(axis=0)    # итог по годам

In [ ]:
load.mean(axis=1)   # среднее по судам

In [ ]:
load.sum()          # дел за все годы

## Итоги

Массив — столбец будущей таблицы: один тип данных, операции сразу над всем столбцом, отбор по маске, статистики и оси.